# Export Ingested Data to Contabo VM

Since the free tier of Wherobots terminates every 3 hours, this notebook provides two methods to save your ingested GeoParquet datasets outside Wherobots directly onto your **Contabo VM**.

## Method 1: Secure SFTP / SSH Transfer (No VM Setup Required)

This method uses the standard SSH server running on your Contabo VM to upload files from the Wherobots local `/tmp/raw` directory with real-time transfer progress.

In [ ]:
!pip install paramiko

In [ ]:
import os
import sys
import paramiko

# Contabo VM Connection Parameters
CONTABO_HOST = "185.208.207.241"
CONTABO_PORT = 22
CONTABO_USER = "root"
CONTABO_PASSWORD = "YOUR_PASSWORD"  # Replace with your root password
REMOTE_DEST_DIR = "/opt/publictransport-crafter/data"  # destination directory on your VM

local_source_dir = "/tmp/raw"

def upload_directory_sftp(local_dir, remote_dir):
    transport = paramiko.Transport((CONTABO_HOST, CONTABO_PORT))
    transport.connect(username=CONTABO_USER, password=CONTABO_PASSWORD)
    sftp = paramiko.SFTPClient.from_transport(transport)
    
    # Ensure remote directory exists
    try:
        sftp.mkdir(remote_dir)
    except IOError:
        pass  # Already exists
        
    for root, dirs, files in os.walk(local_dir):
        for file in files:
            if file.endswith(".parquet") or file.endswith(".crc"):
                local_path = os.path.join(root, file)
                # Calculate remote relative path
                rel_path = os.path.relpath(local_path, local_dir)
                remote_path = os.path.join(remote_dir, rel_path)
                
                # Create remote subdirectories if necessary
                remote_sub = os.path.dirname(remote_path)
                try:
                    sftp.mkdir(remote_sub)
                except IOError:
                    pass
                    
                print(f"\nUploading {rel_path}...")
                
                # Real-time progress callback
                def progress_cb(transferred, total):
                    if total > 0:
                        pct = (transferred / total) * 100
                        sys.stdout.write(f"\r  Progress: {pct:.1f}% ({transferred}/{total} bytes)")
                        sys.stdout.flush()
                
                sftp.put(local_path, remote_path, callback=progress_cb)
                print() # Print newline after file is done
                
    sftp.close()
    transport.close()
    print("\nSFTP Export Completed Successfully!")

# Trigger upload
# upload_directory_sftp(local_source_dir, REMOTE_DEST_DIR)

## Method 2: Direct Spark Writing to MinIO (S3-compatible Object Storage)

If you host a **MinIO** instance on your Contabo VM, you do not need intermediate local files. Spark can write directly to it using the standard `s3a://` protocol.

In [ ]:
from sedona.spark import SedonaContext

# Initialize Sedona Context to point directly to your Contabo MinIO endpoint
config = SedonaContext.builder() \
    .config("spark.hadoop.fs.s3a.endpoint", "http://185.208.207.241:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "YOUR_MINIO_ACCESS_KEY") \
    .config("spark.hadoop.fs.s3a.secret.key", "YOUR_MINIO_SECRET_KEY") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

sedona = SedonaContext.create(config)
print("Spark connected directly to Contabo MinIO S3 backend!")

# Example write:
# df.write.format("geoparquet").save("s3a://your-bucket/nsw_infrastructure_poi.parquet")